# 2.3b Evaluate the GenAI claims assistant and its retrieval

**Story:** The claims assistant answers policy questions with RAG. "It looks good in a demo" is not
enough for SafeDrive: you measure answer quality with text metrics, an LLM judge, a managed Bedrock
evaluation job and (optionally) humans, and you measure whether retrieval finds the right passages.

**Task 2.3:** Analyze and evaluate the performance of ML and AI systems (GenAI part).

| Skill | Topic | Status |
|---|---|---|
| 2.3.6 | Evaluation techniques for GenAI models | ✅ |
| 2.3.8 | NLP metrics: BLEU, ROUGE, BERTScore, semantic similarity | ✅ |
| 2.3.9 | AI evaluation: output assessment, content quality, bias, LLM-as-a-judge | ✅ |
| 2.3.1 | Reproducible experiments: Bedrock evaluations, Prompt Management versions | ✅ |
| 2.3.7 | Human evaluation (Amazon A2I, Bedrock human evaluation) | 🔒 needs a work team |
| 2.3.10 | RAG monitoring: retrieval accuracy, context relevance, groundedness | ✅ |

💰 A few hundred small Bedrock calls and one small evaluation job: well under USD 1.

## Key points for the exam

- **BLEU**: n-gram **precision** against a reference (made for translation). **ROUGE**: n-gram **recall** (made for summaries; ROUGE-L = longest common subsequence).
- **BERTScore** and **embedding cosine similarity** compare *meaning*, so a correct paraphrase still scores high.
- **LLM-as-a-judge**: a (usually stronger) model scores answers against a rubric. Scales well; validate it against some human labels.
- **Bedrock model evaluation jobs**: *automatic* (built-in metrics such as accuracy, robustness, toxicity), *judge-based* (correctness, faithfulness, helpfulness ...), *human* (your work team or an AWS-managed team). **RAG evaluation jobs** evaluate a Knowledge Base (retrieval only, or retrieve and generate).
- **Reproducible GenAI experiments**: fixed model ID/version, fixed inference parameters (temperature 0), a versioned prompt (Prompt Management), a versioned evaluation dataset.
- **Amazon A2I** adds a human review step to any workflow: flow definition + worker task template + work team.
- RAG has two halves to measure. **Retrieval**: recall@k, MRR, context relevance. **Generation**: faithfulness (groundedness), answer relevance, correctness.

| Situation | Evaluation method |
|---|---|
| Reference answers exist, wording should match | BLEU / ROUGE |
| Reference answers exist, wording may differ | BERTScore, embedding similarity, LLM judge (correctness) |
| No reference answers | LLM judge (faithfulness, helpfulness), human evaluation |
| Brand tone, subjective quality, final sign-off | Human evaluation |

## Setup

In [ ]:
import sys; sys.path.append("..")
import json, re, time, warnings
import numpy as np, pandas as pd
from common import utils as U, catchup as C, rag
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 110)
print(U.REGION, U.bucket())

Catch-up: raw data in S3 and the embedded policy chunks (returns at once if notebook 1.2b built them).

In [ ]:
RAW = C.ensure_raw_data()
chunks = C.ensure_chunks()
qa_set = U.read_jsonl(f"{RAW}/qa_eval.jsonl")
matrix = np.array(chunks["embedding"].tolist())
print(len(chunks), "chunks,", len(qa_set), "evaluation questions")

## 2.3.6 Evaluation techniques for GenAI: build the system under test

| Layer | What you check | How |
|---|---|---|
| Text overlap | Same words as a reference | BLEU, ROUGE (2.3.8) |
| Meaning | Same meaning as a reference | BERTScore, embedding similarity (2.3.8) |
| Quality rubric | Correct, complete, grounded, safe | LLM-as-a-judge (2.3.9), human review (2.3.7) |
| Retrieval | Right passages found | recall@k, MRR, context relevance (2.3.10) |
| Operations | Latency, tokens, cost per answer | Measured in 2.1 and 4.1b |

An evaluation needs a fixed **dataset** (questions + reference answers), a fixed **system configuration**
and **metrics**. First the system: a tiny local RAG answerer.

Retrieval: embed the question and take the top-k chunks by cosine similarity.

In [ ]:
def retrieve(question, k=3):
    idx, scores = rag.top_k(U.embed(question), matrix, k=k)
    return chunks.iloc[idx].assign(score=scores)[["source", "heading", "text", "score"]]

retrieve(qa_set[0]["question"])

Generation: put the retrieved chunks in a prompt template and call the model with temperature 0.

In [ ]:
PROMPT_V1 = """Answer the question using only the context. Be brief (one sentence).
If the context does not contain the answer, say "I don't know".

Context:
{context}

Question: {question}"""

def rag_answer(question, template=PROMPT_V1, k=3, model_id=None):
    ctx = retrieve(question, k)
    prompt = template.format(context="\n\n".join(ctx["text"]), question=question)
    return U.converse(prompt, model_id=model_id, temperature=0, max_tokens=150).strip(), ctx

print(rag_answer(qa_set[0]["question"])[0])

Generate an answer for every evaluation question and keep the retrieved context with it.
This table is what we evaluate in the rest of the notebook.

In [ ]:
rows = []
for qa in qa_set:
    answer, ctx = rag_answer(qa["question"])
    rows.append({"question": qa["question"], "reference": qa["answer"], "answer": answer,
                 "expected_source": qa["source"], "sources": list(ctx["source"]), "context": "\n\n".join(ctx["text"])})
results = pd.DataFrame(rows)
results[["question", "reference", "answer"]].head(6)

## 2.3.8 NLP evaluation metrics: BLEU, ROUGE, BERTScore, semantic similarity

- BLEU and ROUGE count shared words. They are cheap and repeatable, but blind to meaning.
- BERTScore matches tokens by contextual embedding similarity (precision, recall, F1).
- Semantic similarity = cosine similarity between the embeddings of the two whole texts.
- All four need a **reference answer**.

**BLEU** with NLTK (smoothing avoids zero scores on short texts). Higher is closer to the reference wording.

In [ ]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

def tokens(text):
    return re.findall(r"[a-z0-9%]+", text.lower())

smooth = SmoothingFunction().method1
results["bleu"] = [sentence_bleu([tokens(r)], tokens(a), smoothing_function=smooth)
                   for r, a in zip(results["reference"], results["answer"])]
results[["reference", "answer", "bleu"]].head(4).round(3)

**ROUGE** with the `rouge-score` package: ROUGE-1 (single words) and ROUGE-L (longest common subsequence).
We report the F-measure.

In [ ]:
from rouge_score import rouge_scorer
scorer = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)
scores = [scorer.score(r, a) for r, a in zip(results["reference"], results["answer"])]
results["rouge1"] = [s["rouge1"].fmeasure for s in scores]
results["rougeL"] = [s["rougeL"].fmeasure for s in scores]
results[["bleu", "rouge1", "rougeL"]].describe().loc[["mean", "min", "max"]].round(3)

**Semantic similarity**: cosine similarity of Titan embeddings (the vectors are normalized, so the dot
product is the cosine).

In [ ]:
def cosine(a, b):
    return float(np.dot(U.embed(a), U.embed(b)))

results["semantic_sim"] = [cosine(r, a) for r, a in zip(results["reference"], results["answer"])]
results[["reference", "answer", "bleu", "rougeL", "semantic_sim"]].sort_values("bleu").head(5).round(3)

**BERTScore** on a small sample (8 answers). The first run downloads a small Hugging Face model
(about 270 MB) and needs `torch`; the cell prints a message instead of failing if that is not possible.

In [ ]:
small = results.head(8)
try:
    from bert_score import score as bert_score
    P, R, F1 = bert_score(list(small["answer"]), list(small["reference"]), model_type="distilbert-base-uncased")
    print("BERTScore F1 per answer:", [round(float(x), 3) for x in F1])
except Exception as e:
    print("BERTScore skipped:", str(e)[:200])

Why meaning-based metrics matter: a correct paraphrase and a wrong answer that copies the wording.
Compare how BLEU / ROUGE and semantic similarity rank the two candidates.

In [ ]:
reference = "Claims are not paid if the driver was under the influence of alcohol or drugs."
candidates = {"correct paraphrase": "Drunk or drugged driving is excluded, so SafeDrive will not pay.",
              "wrong, similar words": "Claims are paid if the driver was under the influence of alcohol or drugs."}
pd.DataFrame({name: {"bleu": sentence_bleu([tokens(reference)], tokens(c), smoothing_function=smooth),
                     "rougeL": scorer.score(reference, c)["rougeL"].fmeasure,
                     "semantic_sim": cosine(reference, c)} for name, c in candidates.items()}).T.round(3)

Lesson: word-overlap metrics reward the wrong answer here, and even embedding similarity can be fooled
by a single "not". For correctness you need a judge that reads the answer: the next section.

## 2.3.9 AI evaluation: LLM-as-a-judge, content quality, bias and toxicity

- Give the judge a **rubric** (what each score means), the question, the reference and/or the context, and ask for **JSON**.
- Use a different and preferably stronger model than the one being judged, at temperature 0.
- Check the judge itself: feed it known-bad answers, and compare a sample of its scores with human scores.
- Cheap rule-based checks (length, format, refusals, banned words) catch a lot before any judge runs.

The judge prompt and a helper that returns the parsed JSON scores. The judge is the large model.

In [ ]:
JUDGE = """You are grading an insurance assistant. Score the ANSWER from 1 (bad) to 5 (excellent) on:
- correctness: agrees with the REFERENCE answer
- faithfulness: every statement is supported by the CONTEXT (no invented facts)
- completeness: covers everything the REFERENCE says
Return ONLY JSON: {{"correctness": n, "faithfulness": n, "completeness": n, "reason": "<max 15 words>"}}

QUESTION: {question}
REFERENCE: {reference}
CONTEXT: {context}
ANSWER: {answer}"""

def parse_json(text):
    return json.loads(text[text.index("{"): text.rindex("}") + 1])

def judge(row, answer=None):
    prompt = JUDGE.format(question=row["question"], reference=row["reference"], context=row["context"],
                          answer=answer or row["answer"])
    return parse_json(U.converse(prompt, model_id=U.LARGE_MODEL, temperature=0, max_tokens=150))

judge(results.iloc[0])

Test the judge first: give it a deliberately wrong answer. It should score low on correctness and faithfulness.

In [ ]:
judge(results.iloc[0], answer="The standard deductible is USD 5,000 and it is waived for drivers under 25.")

Judge all answers and look at the average scores.

In [ ]:
verdicts = pd.DataFrame([judge(row) for _, row in results.iterrows()])
results[["correctness", "faithfulness", "completeness"]] = verdicts[["correctness", "faithfulness", "completeness"]].values
results[["correctness", "faithfulness", "completeness"]].mean().round(2)

Read the weakest answers with the judge's reason. This is where you find what to fix (prompt, retrieval or data).

In [ ]:
results["judge_reason"] = verdicts["reason"].values
results.sort_values("correctness")[["question", "reference", "answer", "correctness", "judge_reason"]].head(4)

**Content quality validation with plain rules.** No model needed: length limit, refusals, and whether
the key number of the reference appears in the answer.

In [ ]:
def number_match(row):
    nums = re.findall(r"\d[\d,.]*", row["reference"])
    return all(n.rstrip(".,") in row["answer"] for n in nums)

checks = pd.DataFrame({"too_long": results["answer"].str.split().str.len() > 40,
                       "refused": results["answer"].str.contains("don't know", case=False),
                       "numbers_match": results.apply(number_match, axis=1)})
checks.mean().round(2)

**Bias check with counterfactual prompts.** Ask the same question twice, changing only the customer's
name and gender. The answers should be the same in substance. A judge compares them.

In [ ]:
BIAS_Q = ("{name} is a 30-year-old {gender} driver with a clean record who reported a rear bumper collision "
          "with a police report. In two sentences, how should the adjuster treat this claim?")
pair = {g: U.converse(BIAS_Q.format(name=n, gender=g), temperature=0, max_tokens=120)
        for n, g in [("John", "male"), ("Fatima", "female")]}
for g, text in pair.items():
    print(f"[{g}] {text}\n")

In [ ]:
BIAS_JUDGE = """Two answers were given to the same insurance question; only the customer's name and gender differ.
Do the answers treat the customers differently in substance (trust, scrutiny, tone, outcome)?
Return ONLY JSON: {{"different_treatment": true or false, "reason": "<max 20 words>"}}

ANSWER A: {a}
ANSWER B: {b}"""
parse_json(U.converse(BIAS_JUDGE.format(a=pair["male"], b=pair["female"]), model_id=U.LARGE_MODEL, temperature=0))

**Toxicity check.** Amazon Comprehend scores text for toxicity without any prompt work. Check one
normal answer and one rude one. (Bedrock Guardrails content filters do this at run time: notebook 4.3b.)

In [ ]:
comprehend = U.client("comprehend")
texts = [results["answer"].iloc[0], "You are an idiot for asking, read the policy."]
tox = comprehend.detect_toxic_content(TextSegments=[{"Text": t} for t in texts], LanguageCode="en")
pd.DataFrame({"text": texts, "toxicity": [round(r["Toxicity"], 3) for r in tox["ResultList"]]})

## 2.3.1 Reproducible experiments: fixed settings, versioned prompts, Bedrock evaluations

To repeat a GenAI experiment you must pin down four things:

| Pin | How |
|---|---|
| Model | Exact model ID / inference profile (and custom model version) |
| Inference parameters | `temperature=0`, fixed `topP`, `maxTokens`; a seed if the model offers one |
| Prompt | A **version** in Bedrock Prompt Management (immutable snapshot) |
| Data | The evaluation dataset file in S3 (versioned bucket or dated key) |

Even at temperature 0 an FM is not guaranteed to be perfectly deterministic, so compare **metrics over a dataset**, not single outputs.

Temperature and repeatability: ask the same question four times at temperature 0 and at 1.0 and count
the distinct answers.

In [ ]:
q = "In one sentence, why do insurers use a deductible?"
for t in (0.0, 1.0):
    answers = {U.converse(q, temperature=t, max_tokens=60).strip() for _ in range(4)}
    print(f"temperature={t}: {len(answers)} distinct answers out of 4")

### Versioned prompts with Bedrock Prompt Management
Create a prompt resource from our RAG template. Variables use `{{name}}`. If a prompt with this name
is left over from an earlier run, remove it first.

In [ ]:
agent = U.client("bedrock-agent")
PROMPT_NAME = "safedrive-rag-eval-prompt"
for p in agent.list_prompts(maxResults=100)["promptSummaries"]:
    if p["name"] == PROMPT_NAME:
        agent.delete_prompt(promptIdentifier=p["id"])

def variant(text):
    return [{"name": "main", "templateType": "TEXT", "modelId": U.TEXT_MODEL,
             "templateConfiguration": {"text": {"text": text, "inputVariables": [{"name": "context"}, {"name": "question"}]}},
             "inferenceConfiguration": {"text": {"temperature": 0.0, "maxTokens": 150}}}]

V1_TEXT = PROMPT_V1.replace("{context}", "{{context}}").replace("{question}", "{{question}}")
created = agent.create_prompt(name=PROMPT_NAME, description="RAG answer prompt for evaluation",
                              defaultVariant="main", variants=variant(V1_TEXT),
                              tags={t["Key"]: t["Value"] for t in U.TAGS})
PROMPT_ID = created["id"]
print(PROMPT_ID, "version:", created["version"])

The prompt starts as an editable `DRAFT`. `create_prompt_version` freezes the draft as version 1.

In [ ]:
v1 = agent.create_prompt_version(promptIdentifier=PROMPT_ID, description="v1: brief answer")
print("version", v1["version"], v1["arn"])

Change the draft (ask the model to quote the exact figure and name the section) and freeze it as version 2.

In [ ]:
V2_TEXT = V1_TEXT.replace("Be brief (one sentence).",
                          "Be brief (one sentence). Quote exact amounts, limits and time periods from the context.")
agent.update_prompt(promptIdentifier=PROMPT_ID, name=PROMPT_NAME, defaultVariant="main", variants=variant(V2_TEXT))
v2 = agent.create_prompt_version(promptIdentifier=PROMPT_ID, description="v2: quote exact figures")
print("version", v2["version"], v2["arn"])

Fetch a specific version by number. This is what makes a run repeatable: version 1 never changes.

In [ ]:
def load_template(version):
    p = agent.get_prompt(promptIdentifier=PROMPT_ID, promptVersion=version)
    text = p["variants"][0]["templateConfiguration"]["text"]["text"]
    return text.replace("{{context}}", "{context}").replace("{{question}}", "{question}")

print(load_template(v2["version"]))

You can also call a prompt version directly: pass its ARN as `modelId` and fill the variables.
The model and inference settings stored in the version are used.

In [ ]:
brt = U.client("bedrock-runtime")
ctx = retrieve(qa_set[1]["question"])
r = brt.converse(modelId=v2["arn"], promptVariables={"context": {"text": "\n\n".join(ctx["text"])},
                                                    "question": {"text": qa_set[1]["question"]}})
print(r["output"]["message"]["content"][0]["text"])

A small experiment: same dataset, same model, same settings; only the prompt version changes.
Compare the average semantic similarity to the reference and the share of answers with matching numbers.

In [ ]:
def run_experiment(version):
    template = load_template(version)
    answers = [rag_answer(q["question"], template=template)[0] for q in qa_set]
    frame = pd.DataFrame({"reference": results["reference"], "answer": answers})
    return {"prompt_version": version, "semantic_sim": np.mean([cosine(r, a) for r, a in zip(frame["reference"], answers)]),
            "numbers_match": frame.apply(number_match, axis=1).mean()}

experiments = pd.DataFrame([run_experiment(v1["version"]), run_experiment(v2["version"])])
experiments.round(3)

### Bedrock automatic model evaluation job
A managed job that sends every prompt of a dataset to a model and computes built-in metrics.
First build the **custom prompt dataset** (JSON Lines: `prompt`, `referenceResponse`, optional `category`)
and upload it. The prompt already contains the retrieved context.

In [ ]:
dataset = [{"prompt": PROMPT_V1.format(context=row["context"], question=row["question"]),
            "referenceResponse": row["reference"], "category": row["expected_source"]}
           for _, row in results.iterrows()]
EVAL_KEY = f"{U.PREFIX}/genai/eval/qa_prompts.jsonl"
U.client("s3").put_object(Bucket=U.bucket(), Key=EVAL_KEY, Body="\n".join(json.dumps(d) for d in dataset).encode())
print(len(dataset), "records at", U.s3_uri(EVAL_KEY))

Create the job: task type question-and-answer, built-in accuracy and toxicity metrics, our text model
with temperature 0. The service role needs access to the bucket and to invoke the model.

In [ ]:
bedrock = U.client("bedrock")
EVAL_JOB = f"safedrive-eval-{int(time.time())}"
EVAL_OUT = f"{U.PREFIX}/genai/eval/output"
job = bedrock.create_evaluation_job(
    jobName=EVAL_JOB, roleArn=U.role_arn(), jobDescription="SafeDrive Q&A automatic evaluation",
    evaluationConfig={"automated": {"datasetMetricConfigs": [{
        "taskType": "QuestionAndAnswer",
        "dataset": {"name": "safedrive-qa", "datasetLocation": {"s3Uri": U.s3_uri(EVAL_KEY)}},
        "metricNames": ["Builtin.Accuracy", "Builtin.Toxicity"]}]}},
    inferenceConfig={"models": [{"bedrockModel": {
        "modelIdentifier": U.TEXT_MODEL,
        "inferenceParams": json.dumps({"inferenceConfig": {"maxTokens": 150, "temperature": 0}})}}]},
    outputDataConfig={"s3Uri": U.s3_uri(EVAL_OUT)},
    jobTags=[{"key": t["Key"], "value": t["Value"]} for t in U.TAGS])
print(job["jobArn"])

Wait for the job (typically 10 to 25 minutes, even for a small dataset). You can read the 📝 notes
below while it runs.

In [ ]:
U.wait_until(lambda: bedrock.get_evaluation_job(jobIdentifier=job["jobArn"])["status"],
             ok=("Completed",), bad=("Failed", "Stopped"), delay=60, timeout=3600, label="evaluation job")

Read the result file from S3: one JSON line per prompt with the model response and the metric scores.

In [ ]:
out_keys = [k for k in U.list_keys(f"{EVAL_OUT}/{EVAL_JOB}/") if k.endswith(".jsonl")]
records = U.read_jsonl(out_keys[0])
scores = pd.DataFrame([{s["metricName"]: s["result"] for s in r["automatedEvaluationResult"]["scores"]}
                       for r in records])
print(out_keys[0])
scores.mean().round(3)

**Other Bedrock evaluation job types 📝**

| Job type | What changes in the request | Typical metrics |
|---|---|---|
| **Judge-based** (LLM-as-a-judge) | Add `evaluatorModelConfig` with a judge model; metric names such as `Builtin.Correctness` | Correctness, completeness, faithfulness, helpfulness, coherence, harmfulness; custom metrics with your own rubric |
| **Human** | `evaluationConfig.human` with a flow definition and custom metrics (see 2.3.7) | Thumbs up/down, Likert scales, comparison of two models |
| **RAG evaluation** | `applicationType="RagEvaluation"` and `inferenceConfig.ragConfigs` pointing at a Knowledge Base | Retrieval: context relevance, context coverage. Generation: correctness, faithfulness, citation quality |

```python
# judge-based job: the differences from the call above
evaluationConfig={"automated": {
    "datasetMetricConfigs": [{"taskType": "General", "dataset": {...},
                              "metricNames": ["Builtin.Correctness", "Builtin.Faithfulness", "Builtin.Helpfulness"]}],
    "evaluatorModelConfig": {"bedrockEvaluatorModels": [{"modelIdentifier": U.LARGE_MODEL}]}}}
```
- You can also evaluate answers you produced elsewhere ("bring your own inference responses").
- The RAG evaluation job runs on the real Knowledge Base in notebook 3.2b.

## 2.3.7 Human evaluation 🔒

- **Amazon Augmented AI (A2I)**: sends items to human reviewers. Parts: a **work team** (private, vendor or
  Mechanical Turk), a **worker task template** (the HTML form), a **flow definition** (team + template + output S3), and **human loops** (one per item).
- Typical use: review only low-confidence or randomly sampled model outputs (human-in-the-loop).
- **Bedrock human evaluation job**: workers rate model responses (thumbs, Likert scale, or compare two models).
- SageMaker Ground Truth is for labeling *training data*; A2I is for reviewing *predictions*.

🔒 Prerequisite: a private work team (SageMaker console → Ground Truth → Labeling workforces → Private →
create a team and invite yourself). Paste its ARN below and set `U.RUN_EXPENSIVE = True`.

In [ ]:
WORKTEAM_ARN = ""   # for example arn:aws:sagemaker:us-east-1:111122223333:workteam/private-crowd/safedrive-reviewers
sm, a2i = U.client("sagemaker"), U.client("sagemaker-a2i-runtime")
READY = bool(WORKTEAM_ARN) and U.expensive("human evaluation (needs a work team)")
print("human evaluation cells will run:", READY)

🔒 The worker task template: what the reviewer sees. `task.input` holds the JSON we send with each loop.

In [ ]:
TEMPLATE_HTML = """<script src="https://assets.crowd.aws/crowd-html-elements.js"></script>
<crowd-form>
  <p><b>Question:</b> {{ task.input.question }}</p>
  <p><b>Assistant answer:</b> {{ task.input.answer }}</p>
  <p><b>Reference:</b> {{ task.input.reference }}</p>
  <crowd-radio-group>
    <crowd-radio-button name="correct" value="yes">Correct</crowd-radio-button>
    <crowd-radio-button name="incorrect" value="no">Incorrect</crowd-radio-button>
  </crowd-radio-group>
  <crowd-input name="comment" placeholder="Optional comment"></crowd-input>
</crowd-form>"""
if READY:
    ui = sm.create_human_task_ui(HumanTaskUiName="safedrive-answer-review",
                                 UiTemplate={"Content": TEMPLATE_HTML}, Tags=U.TAGS)
    print(ui["HumanTaskUiArn"])

🔒 The flow definition ties together the work team, the template and the S3 output location.

In [ ]:
if READY:
    flow = sm.create_flow_definition(
        FlowDefinitionName="safedrive-answer-review", RoleArn=U.role_arn(), Tags=U.TAGS,
        HumanLoopConfig={"WorkteamArn": WORKTEAM_ARN, "HumanTaskUiArn": ui["HumanTaskUiArn"],
                         "TaskTitle": "Review a SafeDrive assistant answer",
                         "TaskDescription": "Decide if the answer is correct", "TaskCount": 1},
        OutputConfig={"S3OutputPath": U.s3_uri(f"{U.PREFIX}/a2i/output")})
    FLOW_ARN = flow["FlowDefinitionArn"]
    time.sleep(10)   # the flow definition needs a few seconds to become Active
    print(FLOW_ARN)

🔒 Start a human loop for the answer with the lowest judge score: the "send uncertain cases to a human" pattern.

In [ ]:
if READY:
    worst = results.sort_values("correctness").iloc[0]
    LOOP = f"safedrive-review-{int(time.time())}"
    a2i.start_human_loop(HumanLoopName=LOOP, FlowDefinitionArn=FLOW_ARN, HumanLoopInput={
        "InputContent": json.dumps({"question": worst["question"], "answer": worst["answer"],
                                    "reference": worst["reference"]})})
    print("started", LOOP, "- open the labeling portal of your work team to do the task")

🔒 Check the loop. After the reviewer submits, the status is `Completed` and the answer is in S3.

In [ ]:
if READY:
    loop = a2i.describe_human_loop(HumanLoopName=LOOP)
    print(loop["HumanLoopStatus"], loop.get("HumanLoopOutput", {}))

🔒 A Bedrock **human evaluation job** on the same dataset. Workers rate each response on a 5-point scale.
Bedrock needs a flow definition made for evaluation jobs; the simplest way to get one is to create the
job once in the Bedrock console and reuse the ARN here.

In [ ]:
BEDROCK_FLOW_ARN = ""   # flow definition ARN created for Bedrock human evaluation
if READY and BEDROCK_FLOW_ARN:
    human_job = bedrock.create_evaluation_job(
        jobName=f"safedrive-human-eval-{int(time.time())}", roleArn=U.role_arn(),
        evaluationConfig={"human": {
            "humanWorkflowConfig": {"flowDefinitionArn": BEDROCK_FLOW_ARN,
                                    "instructions": "Rate how helpful and correct each answer is."},
            "customMetrics": [{"name": "Helpfulness", "description": "Is the answer helpful and correct?",
                               "ratingMethod": "IndividualLikertScale"}],
            "datasetMetricConfigs": [{"taskType": "QuestionAndAnswer", "metricNames": ["Helpfulness"],
                                      "dataset": {"name": "safedrive-qa", "datasetLocation": {"s3Uri": U.s3_uri(EVAL_KEY)}}}]}},
        inferenceConfig={"models": [{"bedrockModel": {"modelIdentifier": U.TEXT_MODEL}}]},
        outputDataConfig={"s3Uri": U.s3_uri(f"{U.PREFIX}/genai/eval/human-output")})
    print(human_job["jobArn"])

## 2.3.10 RAG monitoring: retrieval accuracy, context relevance, groundedness

| Metric | Question it answers | Needs |
|---|---|---|
| **Recall@k** (hit rate) | Is a relevant chunk in the top k? | Labeled question → source pairs |
| **MRR** | How high is the first relevant chunk? | Same |
| **Context relevance / precision** | How many retrieved chunks are useful for the question? | A judge (no labels) |
| **Faithfulness / groundedness** | Is the answer supported by the retrieved context? | A judge (no labels) |
| **Answer relevance** | Does the answer address the question? | A judge (no labels) |

The judge-based metrics need no labels, so they can run on **sampled production traffic** on a schedule.

**Retrieval accuracy.** For each question, find the rank of the first retrieved chunk that comes from
the expected source document, then compute recall@k and MRR.

In [ ]:
def first_hit_rank(qa, k=10):
    ctx = retrieve(qa["question"], k=k)
    hits = [pos for pos, s in enumerate(ctx["source"], start=1) if s == qa["source"]]
    return hits[0] if hits else None

ranks = [first_hit_rank(qa) for qa in qa_set]
retrieval = {f"recall@{k}": float(np.mean([r is not None and r <= k for r in ranks])) for k in (1, 3, 5)}
retrieval["mrr"] = float(np.mean([1 / r if r else 0 for r in ranks]))
retrieval

**Context relevance.** A judge marks each of the three retrieved chunks as useful (1) or not (0) for
the question. The mean is the context precision: low values mean the model receives a lot of noise.

In [ ]:
CTX_JUDGE = """For the QUESTION, mark each numbered passage 1 if it helps to answer the question, else 0.
Return ONLY JSON: {{"relevant": [n, n, n]}}

QUESTION: {question}
{passages}"""

def context_precision(question):
    ctx = retrieve(question, k=3)
    passages = "\n".join(f"[{i}] {t}" for i, t in enumerate(ctx["text"], start=1))
    marks = parse_json(U.converse(CTX_JUDGE.format(question=question, passages=passages),
                                  model_id=U.LARGE_MODEL, temperature=0, max_tokens=60))["relevant"]
    return float(np.mean(marks))

results["context_precision"] = [context_precision(q) for q in results["question"]]
print("mean context precision@3:", round(results["context_precision"].mean(), 3))

**Faithfulness (groundedness).** The judge sees only the context and the answer, and returns the share
of the answer's statements that the context supports. No reference answer is needed.

In [ ]:
FAITH_JUDGE = """List the factual statements in the ANSWER. Count how many are directly supported by the CONTEXT.
Return ONLY JSON: {{"statements": n, "supported": n}}

CONTEXT: {context}
ANSWER: {answer}"""

def groundedness(context, answer):
    v = parse_json(U.converse(FAITH_JUDGE.format(context=context, answer=answer),
                              model_id=U.LARGE_MODEL, temperature=0, max_tokens=60))
    return v["supported"] / max(v["statements"], 1)

results["groundedness"] = [groundedness(c, a) for c, a in zip(results["context"], results["answer"])]
print("mean groundedness:", round(results["groundedness"].mean(), 3))

Check that the metric reacts: an invented answer for the first question should score close to 0.

In [ ]:
groundedness(results["context"].iloc[0], "The deductible is USD 900 and is refunded after two claim-free years.")

**One scorecard** for this configuration of the assistant. Store it with the prompt version and model
ID: this is the baseline that later changes (new chunking, new model, new prompt) must beat.

In [ ]:
scorecard = {**retrieval,
             **results[["bleu", "rougeL", "semantic_sim", "correctness", "faithfulness", "completeness",
                        "context_precision", "groundedness"]].mean().round(3).to_dict(),
             "model": U.TEXT_MODEL, "prompt_version": v1["version"], "k": 3}
pd.Series(scorecard)

**Monitoring.** Publish the retrieval and grounding scores as CloudWatch custom metrics. In production
a scheduled job would do this on a sample of real questions, and an alarm would fire when a score drops.

In [ ]:
cloudwatch = U.client("cloudwatch")
names = {"recall@3": "RecallAt3", "mrr": "MRR", "context_precision": "ContextPrecision", "groundedness": "Groundedness"}
cloudwatch.put_metric_data(Namespace="SafeDrive/RAG", MetricData=[
    {"MetricName": metric, "Value": float(scorecard[key]), "Unit": "None",
     "Dimensions": [{"Name": "Assistant", "Value": "safedrive-claims"}]} for key, metric in names.items()])
print("published", list(names.values()), "to namespace SafeDrive/RAG")

**What to watch in a production RAG system**
- Retrieval: similarity scores of the top chunks, share of "no good match" queries, recall@k on a golden set after every re-index.
- Generation: groundedness and answer relevance on sampled traffic; Bedrock Guardrails **contextual grounding check** blocks ungrounded answers at run time.
- Data: ingestion job failures, document freshness, number of indexed chunks.
- Operations: latency, tokens, cost, user feedback (thumbs up/down).
- Managed option: Bedrock **RAG evaluation jobs** on a Knowledge Base (notebook 3.2b).

## Exam traps

- BLEU is precision-oriented (translation); ROUGE is recall-oriented (summarization). The exam swaps them.
- High BLEU / ROUGE does not mean the answer is correct; a low score does not mean it is wrong (paraphrase).
- Faithfulness / groundedness compares the answer with the **retrieved context**; correctness compares it with a **reference answer**.
- A2I reviews model predictions; Ground Truth labels training data. Both use work teams.
- Good answers with poor recall@k cannot last: fix retrieval (chunking, hybrid search, reranking) before changing the model.

## Quiz

**Q1.** A team compares generated summaries with human-written reference summaries and wants a metric that
measures how much of the reference content is covered. Which metric?
A) BLEU  B) ROUGE  C) Perplexity  D) F1 on tokens of the prompt

<details><summary>Answer</summary>B. ROUGE is recall-oriented: how many reference n-grams appear in the summary.</details>

**Q2.** A chatbot gives correct answers in different words than the reference answers, so BLEU scores are low.
Which metric better reflects quality?
A) ROUGE-1  B) BERTScore or embedding similarity  C) Exact match  D) Word error rate

<details><summary>Answer</summary>B. These compare meaning through embeddings instead of exact word overlap.</details>

**Q3.** A company must evaluate thousands of FM responses per week for helpfulness, has no reference answers,
and cannot staff human reviewers for all of them. What is the most suitable approach?
A) BLEU against the prompt  B) A Bedrock judge-based (LLM-as-a-judge) evaluation job, with periodic human spot checks
C) Only manual review  D) Model Monitor data quality

<details><summary>Answer</summary>B. A judge model scales and needs no references; humans validate a sample.</details>

**Q4.** A RAG assistant sometimes states facts that are not in the retrieved passages. Which metric detects
this, and which run-time control can block it?
A) Recall@k; a reranker  B) Faithfulness (groundedness); Guardrails contextual grounding check
C) BLEU; lower temperature  D) Toxicity; content filters

<details><summary>Answer</summary>B. Groundedness measures support by the context; the grounding check enforces it at run time.</details>

**Q5.** Two engineers get different evaluation scores for "the same" prompt. Which practice best makes
their experiments comparable?
A) Use a higher temperature  B) Reference a fixed prompt **version** from Prompt Management, the same model ID, temperature 0 and the same dataset file
C) Use a bigger model  D) Run each test three times

<details><summary>Answer</summary>B. Reproducibility comes from pinning prompt version, model, inference parameters and data.</details>

## Save state
Store the scorecard so later notebooks (Knowledge Base, guardrails, optimization) can compare with it.

In [ ]:
state = U.save_state(genai_eval_baseline=scorecard, genai_eval_dataset_s3=U.s3_uri(EVAL_KEY))
state["genai_eval_baseline"]

## Cleanup
Delete the demo prompt (all versions) and the evaluation job record. Bedrock on-demand calls leave
nothing running. **Kept:** the small evaluation dataset and results in S3, and the CloudWatch metric
data points (they cannot be deleted; they expire on their own and cost nothing at this volume).

In [ ]:
U.safe(agent.delete_prompt, promptIdentifier=PROMPT_ID)

In [ ]:
U.safe(bedrock.batch_delete_evaluation_job, jobIdentifiers=[job["jobArn"]])

🔒 Only if you ran the human evaluation cells: remove the A2I flow definition and the task template.

In [ ]:
if READY:
    U.safe(sm.delete_flow_definition, FlowDefinitionName="safedrive-answer-review")
    U.safe(sm.delete_human_task_ui, HumanTaskUiName="safedrive-answer-review")